# 05 · Web application & visualisations (Person 5)

**Goal.** This notebook documents the "interface" part of the project:
1. the Streamlit **questionnaire** (17 questions, the 5 question types required by the brief) and the `answers.json` file it produces;
2. how it is plugged into the recommendation engine (`src/recommender.recommend`, P4);
3. the **5 visualisations** of `src/viz.py`: what each one shows and why it makes the result **interpretable**;
4. screenshots of the application (section 6).

**Place in the pipeline (PLAN §3).**
```
Questionnaire (app/streamlit_app.py) ──► answers.json ──► recommend() (P4) ──► UserProfile + top 3 Recommendation
                                                                                     │
                                                         src/viz.py ◄────────────────┘  (radar, bars, map, heatmap)
```

**Run the application** (from the root of the repo):
```bash
python -m scripts.prepare_app      # once: encodes the books + computes the 2D map (avoids waiting during the demo)
streamlit run app/streamlit_app.py
```

In [1]:
# Step 0: imports. The notebook lives in notebooks/; the repo root is added to the path
# so that src/ (P1-P4) and app/ (P5) can be imported.
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio

from app import questions as Q
from src import config, viz
from src.data_loader import load_books
from src.embeddings import EmbeddingEngine

pio.renderers.default = "notebook_connected"   # interactive figures in Jupyter / VS Code
print("Repo root:", ROOT)

Repo root: C:\Users\jadep\ING5\NLP\GenderBook-Projet-NLP


## 1. The questionnaire: 17 questions, 5 question types

The brief requires **open-ended, Likert, guided, multiple-choice and checkbox** questions, with "sufficient free text".
The question content is defined in `app/questions.py` (no Streamlit, so it is testable and reusable by P4/P6); `app/streamlit_app.py` only handles the display.

| App step | Questions | Required type | Streamlit widget | `answers.json` field |
|---|---|---|---|---|
| 1. Tell us about your reading | Q1 (required, ≥ 30 chars), Q2, Q3, Q4 | **Free text** | `st.text_area` | `free_text.*` |
| 2. Genres & themes | Q5 (21 genres), Q6 (20 themes) | **Checkboxes** | `st.checkbox` (+ tooltip = sentence from `blocks.json`) | `genres`, `themes` |
| 3. Context | Q7 era of the story | **Multiple choice** (radio) | `st.radio` | `period` |
| | Q8 audience | **Multiple choice** (list) | `st.selectbox` | `audience` |
| | Q9 formats | Multiple choice | `st.multiselect` | `formats` |
| 4. Your taste | Q10-Q13 complexity, pace, tone, popularity; Q14 importance of the 4 blocks | **Likert 1-5** | `st.slider` + labels at both ends | `likert.*` |
| 5. Guided questions | Q15 favourite author → conditional field; Q16 characters/plot; Q17 exclude the author | **Guided** | `st.radio` + conditional `st.text_input` + `st.checkbox` | `favorite_author`, `guided.*` |

**Design choices:**
- **A single source of truth for the labels**: the Q5/Q6 boxes and the Q7 eras are read from `data/reference/blocks.json` (P2). If P2 changes a label, the interface follows automatically, and P4 directly finds the descriptive sentence to encode.
- **Q7 returns a label of the Atmosphere block** ("20th century", "Future"…): this is the era of the **story**, handled semantically. It is not a filter on `pub_year` (publication year), which is unrelated (see P1's warning in `config.py`).
- **Favourite author checked live** with P1's `find_author` (fuzzy matching): the user immediately sees whether the author is in the catalogue.
- **Multi-step navigation** with `st.session_state`, a progress bar, validation at each step (Q1 required), Back / Next buttons.
- **Saving**: each submission is written to `data/responses/answers_<timestamp>.json` ("store responses in a structured format" requirement) and can be downloaded.

In [2]:
# Step 1: build answers.json for a demo persona, exactly as the app does
blocks = Q.load_blocks()
state = Q.persona_state("Dark academia reader", blocks)   # what the user typed in
print("Validation errors:", Q.validate_all(state))          # [] = valid form
answers = Q.build_answers(state, blocks)
print(json.dumps(answers, indent=2, ensure_ascii=False))

Validation errors: []
{
  "free_text": {
    "ideal_book": "A psychological literary novel about a group of brilliant, isolated students who get drawn into a crime and then have to live with the guilt. I want moral dilemmas, obsession and complex friendships rather than action.",
    "loved_book": "I loved The Secret History by Donna Tartt for its elitist university setting, the slow unravelling of the characters and the feeling of inevitable tragedy.",
    "atmosphere": "Melancholic and autumnal, an old college with libraries, rain and long conversations late at night.",
    "avoid": "No romance-centred plots and no gore."
  },
  "genres": [
    "Mystery & Detective",
    "Literary Fiction"
  ],
  "themes": [
    "Identity & self-discovery",
    "Friendship",
    "Morality & guilt"
  ],
  "period": "20th century",
  "audience": "Adult",
  "formats": [
    "Paperback",
    "Hardback"
  ],
  "favorite_author": "Donna Tartt",
  "guided": {
    "focus": "Characters",
    "standalone_only"

In [3]:
# Step 2: validation blocks a Q1 that is too short (free text is the core of the semantic analysis)
empty = Q.default_state(blocks)
empty["q1_ideal"] = "a good book"
Q.validate_step(0, empty)

['Q1 is required: please write at least 30 characters (11 so far). The more you describe, the better the semantic match.']

## 2. Calling the recommendation engine

The app calls `src.recommender.recommend(answers, top_k=3)` (contract PLAN §4.4). While P4's module is not available,
it uses `app/stub_recommender.py`, which returns objects **in the same format** (`UserProfile`, `Recommendation`),
computed with a simplified formula on the real data (P3's embeddings, P2's `label_scores`, P1's filters).
The switch is automatic: `src/recommender.py` just has to exist.

> ⚠️ If the next cell prints `stub`, the scores below are **not** those of the final project: re-run the notebook once P4 is integrated.

In [4]:
# Step 3: final engine (P4) if integrated, otherwise EmbeddingEngine (P3) + stub.
# First launch: model download + encoding of the 12,000 books (cached afterwards).
try:
    from src.recommender import get_recommender, recommend   # final engine (P4)
    MODE = "P4"
    reco = get_recommender()
    engine, books = reco.engine, reco.books
    profile, recs = recommend(answers, top_k=3)
except ImportError:
    from app.stub_recommender import recommend                # temporary stand-in (same contract)
    MODE = "stub"
    engine = EmbeddingEngine(books=load_books())
    books = engine.books
    profile, recs = recommend(answers, top_k=3, engine=engine, books=books)
matrix = engine.book_matrix()

print("Mode:", MODE)
print("Profile:", profile.summary)
pd.DataFrame([{"rank": r.rank, "title": r.title, "authors": r.authors, "final_score": round(r.final_score, 3),
               "match": r.match_label, "coverage": round(r.coverage, 3)} for r in recs])

Could not reach the Hub (Server disconnected without sending a response.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'Server disconnected without sending a response.' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].


Mode: stub
Profile: You are looking for literary fiction, mystery & detective books exploring morality & guilt and identity & self-discovery, with a character-driven style and a dark & unsettling atmosphere.


,rank,title,authors,final_score,match,coverage
0,1,Escape,Barbara Delinsky,0.882,Excellent,0.851
1,2,PostSecret : Extraordinary Confessions From Or...,Frank Warren,0.830,Excellent,0.682
2,3,The Infatuations,Javier Marias,0.825,Excellent,0.879


## 3. Visualisations (`src/viz.py`)

All the functions of `viz.py` are **pure**: they take data that has already been computed and return a Plotly figure.
The same function is used in the app, in this notebook and for the report figures (PNG export).
**Shared colour scheme**: one fixed colour per block in every figure (Genre blue, Themes green, Style orange, Atmosphere purple, synopsis grey, penalty red), so that the reader can link one chart to the next and to the label chips of the recommendation cards.

### 3.1 Reader profile radar — `radar_profile(profile.block_affinities)`
**What it shows:** for each of the 4 blocks, the labels closest to the user's answers (cosine between their text and the label's descriptive sentence).
**Why it is interpretable:** the user can check at a glance that the system "understood" the request (e.g. a request about an old university brings out *Academic & campus* without the word ever being written). One radar per block rather than a single one: ~66 labels on one radar would be unreadable, and splitting them reflects the structure of the label set.

In [5]:
fig_profile = viz.radar_profile(profile.block_affinities)
fig_profile.show()

### 3.2 "You vs the books" radar — `radar_compare(profile.weights, {title: block_scores})`
**What it shows:** dashed, the importance the user gives to each block (Likert Q14, mapped to [0, 1]); in colour, each book's calibrated score on that block (and on the synopsis similarity).
**How to read it:** when the book's polygon covers the user's, the book meets the criteria the user cares about; a dip on an important axis explains a lower score. It is the visual translation of the **coverage score** Σ W_b·S_b / Σ W_b.

In [6]:
fig_compare = viz.radar_compare(profile.weights, {r.title: r.block_scores for r in recs})
fig_compare.show()

### 3.3 Score breakdown — `score_breakdown(recs)`
**What it shows:** each bar is a book, each segment the contribution of one component (weight × calibrated similarity), the "avoid" penalty goes to the left (in red), the black diamond is the final score.
**Why it is essential:** the brief asks us to "demonstrate how results are derived from the semantic analysis". The segments add up to the final score: the recommendation is **traceable** end to end. It also shows immediately which criterion made a book win.

In [7]:
fig_breakdown = viz.score_breakdown(recs)
fig_breakdown.show()

# Check: the sum of the contributions = the final score (up to the [0, 1] clipping)
for r in recs:
    print(f"{r.title[:40]:40s}  sum of contributions = {sum(r.contributions.values()):.3f}   final score = {r.final_score:.3f}")

Escape                                    sum of contributions = 0.882   final score = 0.882
PostSecret : Extraordinary Confessions F  sum of contributions = 0.830   final score = 0.830
The Infatuations                          sum of contributions = 0.825   final score = 0.825


### 3.4 Semantic map of the catalogue — `similarity_map(coords, genres, titles, ...)`
**What it shows:** the 12,000 books projected to 2D from their SBERT embeddings (UMAP, cosine metric; PCA fallback if `umap-learn` is missing), coloured by main genre. The user (✖) is placed at the weighted barycentre of their 15 closest books, the top 3 is marked with stars and the nearest neighbours with circles.
**Interpretation:** two nearby books tell stories with a similar meaning, even without shared words. Genres form coherent regions, which visually validates the embeddings; the user falls in the region that matches their request.
**Technical choice:** placing the user with `umap.transform` would require keeping the fitted UMAP model (heavy) and would give an unstable position; the nearest-neighbour barycentre is simple, deterministic and faithful to the idea "you are where the books that resemble you are".
The coordinates are computed only once and cached in `data/embeddings/umap_2d.npy` (+ `umap_book_ids.json` to guarantee the alignment, same files as P3's `src/semantic_map.py`).

In [8]:
coords = viz.load_or_compute_map_coords(matrix, books["book_id"].astype(str).tolist())
q = profile.query_vecs.get("global")
if q is None:                                   # in case P4 does not return the vector
    q = engine.encode([profile.query_texts["global"]])[0]
sims = matrix @ np.asarray(q).ravel()
user_xy, nn = viz.user_map_position(coords, sims, k=15)

idx = {b: i for i, b in enumerate(books["book_id"].astype(str))}
fig_map = viz.similarity_map(
    coords, [viz.primary_genre(g) for g in books["genres"]], books["title"].tolist(),
    highlight=[idx[r.book_id] for r in recs if r.book_id in idx], highlight_labels=[r.title for r in recs],
    user_xy=user_xy, neighbours=nn,
)
fig_map.show()

### 3.5 Similarity heatmap — `similarity_heatmap(sim, rows, columns)`
**What it shows:** the cosine between the **user's words** (rows: answers Q1-Q3 and preference sentences per block) and **what the NLP enrichment assigned to the book** (columns: its genre, theme, style and atmosphere labels, represented by the same vectors as in the engine, plus the closest sentence of the synopsis).
**Why it proves the semantic analysis:** a dark cell links two different phrasings of the same idea. This is exactly what the brief calls "not purely keyword matching".

In [9]:
book = books.iloc[idx[recs[0].book_id]]
ft = answers["free_text"]
rows = [(f"Q{i}: {ft[k]}", ft[k]) for i, k in ((1, "ideal_book"), (2, "loved_book"), (3, "atmosphere")) if ft[k]]
# preference sentences per block (P4: profile.block_phrases; at most 3 per block to stay readable)
phrases = getattr(profile, "block_phrases", None) or {}
for b in config.BLOCKS:
    items = phrases.get(b) or [t.strip() for t in (profile.query_texts.get(b) or "").replace(" | ", ";").split(";") if t.strip()]
    rows += [(f"{b}: {t}", t) for t in items[:3]]

cols, vecs = [], []
for b, col in (("Genre", "genres"), ("Themes", "themes"), ("Style", "styles"), ("Atmosphere", "atmospheres")):
    names, mat = engine.label_matrix(b)
    for label in book[col]:
        cols.append(f"{b} · {label}")
        vecs.append(mat[names.index(label)])
cols.append("Closest sentence")
vecs.append(engine.encode([recs[0].best_sentence])[0])

sim = engine.encode([t for _, t in rows]) @ np.vstack(vecs).T
fig_heat = viz.similarity_heatmap(sim, [r for r, _ in rows], cols, title=f"Your words × {recs[0].title}")
fig_heat.show()

## 4. Explainability in the recommendation cards

Each card of the top 3 shows the **4 required elements** (item, score, main criteria, explanation):
- **the item**: cover, title, authors;
- **the score**: gauge of the final score + *Excellent / Good / Partial match* badge (thresholds 0.75 / 0.55 from `config.MATCH_THRESHOLDS`) + coverage score;
- **the main criteria**: coloured chips of the shared labels with their similarity (`matched_labels`);
- **the explanation** generated by template (`explanation`, P4), preceded by the **synopsis sentence closest** to the request (`best_sentence`), also highlighted in the full synopsis;
- an expandable **"How was this score computed?"** panel: the formula (LaTeX) and a score / weight / contribution table for each component.

In [10]:
for r in recs:
    print(f"#{r.rank} {r.title} — {r.authors}  [{r.match_label}, {r.final_score:.0%}]")
    print("   Closest sentence:", r.best_sentence[:150])
    print("   Criteria:", {b: [l for l, _ in v] for b, v in r.matched_labels.items()})
    print("   Explanation:", r.explanation[:300], "\n")

#1 Escape — Barbara Delinsky  [Excellent, 88%]
   Closest sentence: A first-rate storyteller who creates believable, sympathetic characters who seem as familiar as your neighbors.
   Criteria: {'Genre': ['Literary Fiction'], 'Themes': ['Identity & self-discovery'], 'Style': ['Character-driven', 'Slow and contemplative'], 'Atmosphere': ['Small town']}
   Explanation: Its synopsis is semantically close to what you described (closest sentence, similarity 0.39). It matches your themes: Identity & self-discovery (0.48). It matches your genre: Literary Fiction (0.45). Its atmosphere: Small town (0.29). It matches your style: Character-driven (0.42), Slow and contempl 

#2 PostSecret : Extraordinary Confessions From Ordinary Lives — Frank Warren  [Excellent, 83%]
   Closest sentence: Your secret can be a regret, fear, betrayal, desire, confession, or childhood humiliation.
   Criteria: {'Genre': ['Literary Fiction'], 'Themes': ['Identity & self-discovery', 'Morality & guilt'], 'Style': ['Slow

## 5. Exporting the figures for the report and the slides

`viz.save_figure` writes high-resolution PNGs to `docs/figures/` (prefix `05_`) through **kaleido**.
With kaleido ≥ 1.0, Chrome is needed: if the export fails, run `plotly_get_chrome` once.

In [11]:
paths = viz.export_all({
    "05_radar_profile": fig_profile,
    "05_radar_compare": fig_compare,
    "05_score_breakdown": fig_breakdown,
    "05_similarity_map": fig_map,
    "05_similarity_heatmap": fig_heat,
})
paths

[WindowsPath('C:/Users/jadep/ING5/NLP/GenderBook-Projet-NLP/docs/figures/05_radar_profile.png'),
 WindowsPath('C:/Users/jadep/ING5/NLP/GenderBook-Projet-NLP/docs/figures/05_radar_compare.png'),
 WindowsPath('C:/Users/jadep/ING5/NLP/GenderBook-Projet-NLP/docs/figures/05_score_breakdown.png'),
 WindowsPath('C:/Users/jadep/ING5/NLP/GenderBook-Projet-NLP/docs/figures/05_similarity_map.png'),
 WindowsPath('C:/Users/jadep/ING5/NLP/GenderBook-Projet-NLP/docs/figures/05_similarity_heatmap.png')]

## 6. Screenshots of the application

Screenshots generated automatically by `python -m scripts.capture_screens` (the app must be running in another terminal; requirements: `pip install playwright && python -m playwright install chromium`).

| Screen | What it shows |
|---|---|
| ![](../docs/figures/05_screen_1_free_text.png) | **Step 1**: 4 open-ended questions, Q1 required with a character counter, examples as placeholders. Sidebar: progress, navigation, demo mode. |
| ![](../docs/figures/05_screen_2_genres_themes.png) | **Step 2**: genre and theme checkboxes; the ⓘ tooltip shows the descriptive sentence actually encoded by the engine. |
| ![](../docs/figures/05_screen_3_context.png) | **Step 3**: era of the story (radio), audience (selectbox), formats (multiselect). |
| ![](../docs/figures/05_screen_4_likert.png) | **Step 4**: Likert scales with labels at both ends; importance of the 4 blocks (weights W_b). |
| ![](../docs/figures/05_screen_5_guided.png) | **Step 5**: guided questions; the favourite author is looked up live in the catalogue. |
| ![](../docs/figures/05_screen_6_top3.png) | **Results, top 3**: one card per book (score, badge, closest sentence, explanation, criteria), radar and score breakdown. |
| ![](../docs/figures/05_screen_7_your_profile.png) | **Reader profile**: summary, radars per block, weights, filters applied. |
| ![](../docs/figures/05_screen_7_compare.png) | **Comparison** of the 3 books: stacked bars + shared radar. |
| ![](../docs/figures/05_screen_7_semantic_map.png) | **Semantic map** of the catalogue with the user and the top 3. |
| ![](../docs/figures/05_screen_7_under_the_hood.png) | **Under the hood**: heatmap of the user's words × the book's labels. |

## 7. Robustness and usability choices
- **Heavy resources loaded once** (`st.cache_resource`): SBERT model, catalogue, book matrix, 2D map. Explicit spinner on first launch; `scripts/prepare_app.py` pre-computes everything before the demo.
- **No result after filtering** (audience + format + excluded author too restrictive): clear message and a "Relax the filters" button.
- **Engine unavailable** (no Internet on first launch, error in P4): the app does not crash, it falls back to the stub or disables the map with a message.
- **Demo mode**: 3 pre-filled personas (sidebar, or direct link `?persona=Young%20fantasy%20fan`) for a reliable demo during the presentation.
- **Persistence**: answers survive navigation between steps, can be reloaded from an `answers.json`, and results can be downloaded as JSON (useful for P6's evaluation).